# Exercises — Chapter 10: Portfolio Optimization and Quantitative Trading

Complete the exercises from Lecture 10.

In [ ]:
# Your code here

## Data Lab — SEC EDGAR

Construct fundamental-factor signals from EDGAR XBRL data and back-test simple portfolios.

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course instructor@dauphine.eu"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

LM_POS = {"profitable","strong","growth","improved","exceeded","innovative",
           "efficient","leading","record","gained","successful","robust",
           "increased","advancing","outperformed","benefit","enhanced","positive","achieved","favourable"}
LM_NEG = {"risk","loss","uncertain","decline","adverse","default","impair",
           "volatile","litigation","breach","failed","reduced","weakness",
           "debt","distress","penalty","violation","exposure","downturn","writedown"}


### Exercise [B]: Fundamental Factor Dashboard

In [ ]:
# --- Exercise [B]: Fundamental Factor Dashboard ---
TICKERS_10 = ["AAPL","MSFT","GOOGL","JPM","BAC","XOM","JNJ","PFE","GE","F"]

rows_b = []
for t in TICKERS_10:
    try:
        cik = get_cik(t)
        def gv(c, fb=None):
            try: return get_annual_series(cik, c).sort_values("date").iloc[-1][c]
            except: return fb
        A   = gv("Assets")       or 1
        E   = gv("StockholdersEquity") or 1
        NI  = gv("NetIncomeLoss") or 0
        D   = gv("LongTermDebt")  or 0
        Rev = gv("Revenues")      or 1
        # Revenue growth: compare most recent two years
        try:
            rev_s = get_annual_series(cik,"Revenues").sort_values("date")["Revenues"]
            rev_g = (rev_s.iloc[-1]/rev_s.iloc[-2]-1) if len(rev_s)>=2 else 0
        except: rev_g = 0
        rows_b.append({"ticker":t,"ROE":round(NI/abs(E),3),"Leverage":round(D/A,3),"Rev_Growth":round(rev_g,3)})
    except Exception as e:
        print(f"{t}: {e}")

df_10b = pd.DataFrame(rows_b).set_index("ticker").sort_values("ROE", ascending=False)
print(df_10b)

### Exercise [I]: Simple Value Factor Portfolio

In [ ]:
# --- Exercise [I]: Value Factor Portfolio ---
rows_pb = []
for t in TICKERS_10:
    try:
        cik = get_cik(t)
        bv  = get_annual_series(cik,"StockholdersEquity").sort_values("date").iloc[-1]["StockholdersEquity"]
        try:
            import yfinance as yf
            info   = yf.Ticker(t).info
            shares = info.get("sharesOutstanding") or info.get("impliedSharesOutstanding") or 1
            price  = info.get("currentPrice") or info.get("regularMarketPrice") or 1
            bvps   = bv / shares
            pb     = price / max(bvps, 0.01)
        except: pb = None
        if pb and pb > 0:
            rows_pb.append({"ticker":t,"pb":pb,"bv_b":bv/1e9})
    except Exception as e:
        print(f"{t}: {e}")

df_pb = pd.DataFrame(rows_pb).set_index("ticker").sort_values("pb")
print(df_pb)
value_port  = list(df_pb.head(3).index)
growth_port = list(df_pb.tail(3).index)
print(f"Value portfolio:  {value_port}")
print(f"Growth portfolio: {growth_port}")

# 1-year returns
def port_return(tickers):
    rets = []
    for t in tickers:
        try:
            import yfinance as yf
            hist = yf.download(t, period="1y", progress=False, auto_adjust=True)["Close"]
            if len(hist) >= 2: rets.append(float(hist.iloc[-1]/hist.iloc[0]-1))
        except: pass
    return np.mean(rets) if rets else None

rv = port_return(value_port)
rg = port_return(growth_port)
print(f"\n1-year return — Value: {rv*100:.1f}%  Growth: {rg*100:.1f}%" if rv and rg else "Returns unavailable")

### Exercise [A]: Annual Factor Backtest

In [ ]:
# --- Exercise [A]: Annual Factor Backtest (simplified) ---
print("Note: Full multi-year P/B backtest requires historical XBRL data.")
print("This scaffold uses the 1-year value portfolio return and projects it forward,")
print("while computing vol and drawdown from actual daily price data.")

if rv is not None:
    import yfinance as yf

    ann_ret   = rv
    risk_free = 0.02
    n_years   = 5

    # Annualised vol from actual daily returns over the past year
    daily_rets = []
    for t in value_port:
        try:
            hist = yf.download(t, period="1y", progress=False, auto_adjust=True)["Close"]
            dr = hist.pct_change().dropna()
            daily_rets.extend(dr.values.tolist())
        except:
            pass
    ann_vol = np.std(daily_rets) * np.sqrt(252) if daily_rets else abs(ann_ret)
    sharpe  = (ann_ret - risk_free) / max(ann_vol, 0.001)

    # Max drawdown from equal-weight daily portfolio over the past year
    port_price = None
    n_stocks = 0
    for t in value_port:
        try:
            hist = yf.download(t, period="1y", progress=False, auto_adjust=True)["Close"].dropna()
            normed = hist / hist.iloc[0]
            port_price = normed if port_price is None else port_price + normed
            n_stocks += 1
        except:
            pass
    if port_price is not None and n_stocks > 0:
        port_price   = port_price / n_stocks
        rolling_peak = port_price.cummax()
        max_dd       = float(((port_price - rolling_peak) / rolling_peak).min())
    else:
        max_dd = float("nan")

    # Stylised 5-year growth chart (constant-return projection for illustration)
    cum_values = [1.0]
    for _ in range(n_years):
        cum_values.append(cum_values[-1] * (1 + ann_ret))

    print(f"\nAnnualised return: {ann_ret*100:.1f}%")
    print(f"Annualised vol:    {ann_vol*100:.1f}%  (from actual daily returns)")
    print(f"Sharpe ratio:      {sharpe:.2f}")
    print(f"Max drawdown:      {max_dd*100:.1f}%  (actual 1-year path)")

    # Equal-weight benchmark (same 1-year projection)
    bm_ret  = port_return(TICKERS_10[:6]) or 0.08
    bm_vals = [1.0]
    for _ in range(n_years):
        bm_vals.append(bm_vals[-1] * (1 + bm_ret))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(n_years + 1), cum_values, "o-b", label="Value portfolio")
    ax.plot(range(n_years + 1), bm_vals,    "s--r", label="Equal-weight benchmark")
    ax.set_xlabel("Year")
    ax.set_ylabel("Portfolio value ($1 invested)")
    ax.set_title("Annual Factor Backtest (Stylised — constant-return projection)")
    ax.legend()
    plt.tight_layout()
    plt.show()